In [73]:
import matplotlib.pyplot as plt
from typing import List
import glob
import os
import numpy as np
import pandas as pd 
# import seaborn as sns
import plotly.graph_objects as go
import numpy as np
import plotly.io as pio


import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# Coleta dos dados

In [74]:
results_flows_directories = glob.glob('../../results/results_flows*/*')
results_flows_directories

['../../results/results_flows/msf_s_50_p_4_a_1.0_c_0',
 '../../results/results_flows/kuririnPPO_s_50_p_4_a_1.0_c_0']

In [75]:
metricas_de_coleta = ['tempo',
                        'cpu_utilization',
                        'cache_utilization',
                        'bandwidth_utilization',
                        'success',
                        'latency',
                        'decision_time_ms',
                        'running_sfcs',
                        'cpu_saved',
                        'shared_vnfs']

In [76]:
big_data = pd.DataFrame()

def colect_data_from_alg_directory(results_flows_directories):
    data_nla = []

    for alg_dir in results_flows_directories:
        simulacoes_okays = 0
        simu_exec_name = alg_dir.split('/')[-1].split('_')
        share = 'y'
        alg_name = simu_exec_name[0].split("\\")[0]
        print(alg_name)
        if alg_name == 'g':
            alg_name = 'Greedy'
        if alg_name == 'ga':
            alg_name =  'GA'
        if alg_name == 'msf':
            alg_name = 'MSF'
        if alg_name == 'goku':
            alg_name = 'OSCIM'
        if alg_name == 'vegeta':
            alg_name = 'Resilient-OSCIM'
        if alg_name == 'musfico':
            alg_name = 'MuSFiCO'
        if alg_name == 'greedyb':
            alg_name = 'GreedyB'
        if alg_name == 'kuririnPPO':
            alg_name = 'Kuririn PPO'

        files = os.listdir(alg_dir)
        print(f"Simulação:{alg_name}") 
        print("Quantidade de csv: ",len(files))
        
        for file in files:
            data_path = os.path.join(alg_dir, file)
            try:
                simulation_df = pd.read_csv(data_path)
            except:
                continue
        
            primeiro_tempo = simulation_df['timestamp'].values[0]
            simulation_df['tempo'] = simulation_df[['timestamp']].applymap(lambda x: x - primeiro_tempo)
            simulation_is_success = 'sfc_cache_p4_50' in simulation_df['sfc_id'].values
            
            if simulation_is_success: 
                simulacoes_okays = simulacoes_okays + 1
                simulation_df = simulation_df[metricas_de_coleta]
                
                #Arrendondar tempo
                simulation_df['tempo'] = simulation_df['tempo'].astype(int)
                
                simulation_df = simulation_df.replace('None', pd.NA)

                latency_col = simulation_df[['tempo','latency']] 
                latency_col.dropna(inplace=True)
                latency_col.loc[:, 'latency'] = latency_col['latency'].astype(float)


                ###############################################
                cumulative_sum_success = 0
                cumulative_avg_success = []
                for i, value in enumerate(simulation_df['success']):
                    cumulative_sum_success += value
                    cumulative_avg_success.append(cumulative_sum_success / (i + 1))
                simulation_df['success'] = cumulative_avg_success
                ###############################################

                simulation_df = simulation_df.groupby('tempo', as_index=False).mean(numeric_only=True)
                latency_df = latency_col.groupby('tempo',as_index=False).mean(numeric_only=True).reset_index()
                
                tempo_range = simulation_df['tempo'].max()
                df_mean = simulation_df.set_index('tempo').reindex(range(tempo_range + 1))
                df_mean = df_mean.fillna(method='ffill')
                df_mean = df_mean.reset_index()


                latency_df = latency_df.set_index('tempo').reindex(range(tempo_range + 1))
                latency_df = latency_df.fillna(method='ffill')
                latency_df = latency_df.reset_index()
         
                df_mean['latency'] = latency_df['latency']
                df_mean['algorithm'] = alg_name
                #df_mean['sharing'] = share
                df_mean= df_mean.iloc[0:1000]
                data_nla.append(df_mean)

        data_nla_f = pd.concat(data_nla)
        print("Simulações de sucesso: ",simulacoes_okays)
        print("Dados Nulos: ", data_nla_f.isnull().sum().sum())
        print()

    return data_nla_f

In [77]:
big_data = colect_data_from_alg_directory(results_flows_directories)

msf
Simulação:MSF
Quantidade de csv:  28
Simulações de sucesso:  27
Dados Nulos:  0

kuririnPPO
Simulação:Kuririn PPO
Quantidade de csv:  37
Simulações de sucesso:  37
Dados Nulos:  0



In [78]:
big_data

,tempo,cpu_utilization,cache_utilization,bandwidth_utilization,success,latency,decision_time_ms,running_sfcs,cpu_saved,shared_vnfs,algorithm
0,0,0.05415,0.0275,0.013925,1.000000,9.1525,4.938125,0.0,0.0,0.0,MSF
1,1,0.05415,0.0275,0.013925,1.000000,9.1525,4.938125,0.0,0.0,0.0,MSF
2,2,0.05415,0.0275,0.013925,1.000000,9.1525,4.938125,0.0,0.0,0.0,MSF
3,3,0.05415,0.0275,0.013925,1.000000,9.1525,4.938125,0.0,0.0,0.0,MSF
4,4,0.05415,0.0275,0.013925,1.000000,9.1525,4.938125,0.0,0.0,0.0,MSF
...,...,...,...,...,...,...,...,...,...,...,...
995,995,0.36340,0.1650,0.204600,0.889878,9.4300,10.841000,0.0,0.0,0.0,Kuririn PPO
996,996,0.36280,0.1925,0.199700,0.890234,9.7800,9.589000,0.0,0.0,0.0,Kuririn PPO
997,997,0.36280,0.1925,0.199700,0.890234,9.7800,9.589000,0.0,0.0,0.0,Kuririn PPO
998,998,0.36280,0.1925,0.199700,0.890234,9.7800,9.589000,0.0,0.0,0.0,Kuririn PPO


In [79]:
big_data["CPU Salva por SFC"] =  big_data["cpu_saved"] / big_data["running_sfcs"] 
big_data["CPU Salva por Servidor"] =  big_data["cpu_saved"] / 35 

In [80]:
# Suposições
capacidade_maxima_banda_gbps = 10  # Capacidade máxima da banda em Gbps

# Calculando métricas
big_data["eficiencia de cpu"] =  big_data["cpu_utilization"] / big_data["running_sfcs"] 

big_data["eficiencia de banda"] =   big_data["bandwidth_utilization"]   / big_data["running_sfcs"] 
big_data["eficiencia de cache"] =  big_data["cache_utilization"]  / big_data["running_sfcs"] 

# big_data["bit_rate"] = big_data["practical_bandwidth_utilization"] * capacidade_maxima_banda_gbps

# Função para calcular a pontuação da latência
def calcular_pontuacao_latencia(latencia):
    if pd.isna(latencia):
        return 0  # Latência Nula
    elif latencia > 6:
        return -1  # Latência Ruim
    else:
        return 2  # Latência Boa

# Função para calcular a pontuação da aceitação
def calcular_pontuacao_success(success):
    # Convertendo a taxa de sucesso para uma escala de 0 a 1 e multiplicando por 10 para obter uma pontuação máxima de 10
    return success * 10

# Aplicando as funções para calcular as pontuações
big_data['pontuacao_latencia'] = big_data['latency'].apply(calcular_pontuacao_latencia)
big_data['pontuacao_success'] = big_data['success'].apply(calcular_pontuacao_success)

# Calculando a métrica final de qualidade do serviço
big_data['QoS'] = big_data['pontuacao_latencia'] + big_data['pontuacao_success']

In [81]:
# Lista de algoritmos a serem analisados
# algoritmos = ['GreedyB', 'Kuririn PPO']
algoritmos = ['Kuririn PPO', "MSF"]

# Dicionário para armazenar os dados processados de cada algoritmo
dados_processados = {}

def process_data(data):
    data = data.groupby('tempo').mean()
    return data

for alg in algoritmos:
    # Filtrando os dados baseado no algoritmo e na condição de compartilhamento
    dados_filtrados = big_data[(big_data['algorithm'] == alg)]

    # Removendo as colunas 'algor}ithm' e 'sharing'
    dados_filtrados = dados_filtrados.drop(['algorithm'], axis=1)

    # Processando os dados filtrados
    dados_processados[alg] = process_data(dados_filtrados)


In [82]:
dados_processados

{'Kuririn PPO':        cpu_utilization  cache_utilization  bandwidth_utilization   success  \
 tempo                                                                        
 0             0.054641           0.030659               0.022236  1.000000   
 1             0.054641           0.030659               0.022236  1.000000   
 2             0.054641           0.030659               0.022236  1.000000   
 3             0.054641           0.030659               0.022236  1.000000   
 4             0.054641           0.030659               0.022236  1.000000   
 ...                ...                ...                    ...       ...   
 995           0.384606           0.227917               0.201207  0.913472   
 996           0.378181           0.228542               0.196819  0.913578   
 997           0.379183           0.229792               0.196372  0.913433   
 998           0.375683           0.229821               0.195737  0.912158   
 999           0.380532           0.2

In [83]:
# Agora, dados_processados contém os dados processados para cada algoritmo
# Acessando os dados processados para cada algoritmo:
# greedyb_data_share = dados_processados['GreedyB']
kuririnPPO_data_share = dados_processados['Kuririn PPO']
msf_data_share = dados_processados['MSF']

# res_oscim_data_share = dados_processados['Resilient-OSCIM']

In [84]:
# print(greedyb_data_share['decision_time_ms'].mean())

print(msf_data_share['decision_time_ms'].mean())
print(kuririnPPO_data_share['decision_time_ms'].mean())

# print(musfico_data_share['decision_time_ms'].mean())
# print(gr_data_share['decision_time_ms'].mean())

4.723195884997885
8.93495918208109


# Plot de linha

In [85]:
x = kuririnPPO_data_share.index.values

In [86]:
metricas = {
    "Taxa de Aceitação (%)": "success",
    
    "CPU (%)": "cpu_utilization",

    "Largura de Banda (%)": "bandwidth_utilization",

    "Latência (ms)": "latency",
    "Tempo de Decisão(s)": "decision_time_ms",

    "SF's Compartilhadas": "shared_vnfs",
    #"Bit Rate (Gbps)": "bit_rate",
    #"Quality of Service (QoS)": "QoS",
    #"Running SFC's":"running_sfcs"
}


# Nomes dos algoritmos para legendas
legendas = {
    'MSF': 'MSF',
    'Kuririn PPO': 'Kuririn PPO',

    # 'Resilient-OSCIM': 'Resilient-OSCIM',
}

In [87]:
# import kaleido
import plotly.graph_objects as go
import numpy as np
import pandas as pd

def plotly_three_lines_graph_with_error_bars(y2, y3, y4="sla", xaxis_title='Tempo', yaxis_title='Y Axis', linha2='linha2', linha3='linha3', linha4='linha4', steps=1, x_scale_factor=100):
    # Calcular média
    y2_mean = y2.groupby(np.arange(len(y2))//steps).mean()
    y3_mean = y3.groupby(np.arange(len(y3))//steps).mean()
    #y4_mean = y4.groupby(np.arange(len(y4))//steps).mean()
    
    # Calcular desvio padrão
    y2_std = y2.groupby(np.arange(len(y2))//steps).std()
    y3_std = y3.groupby(np.arange(len(y3))//steps).std()
    #y4_std = y4.groupby(np.arange(len(y4))//steps).std()
    
    # Criar índices para o eixo X e aplicar transformação de escala
    index = np.arange(0, len(y2), steps) #/ x_scale_factor  # Transformação de escala aplicada aqui
    
    # Ajustar título do eixo X para refletir a reescalação
    scale_info = f" (s)"
    adjusted_xaxis_title = xaxis_title + scale_info

    # Plot
    fig = go.Figure()
    
    # Adicionar linhas e barras de erro
    fig.add_trace(go.Scatter(x=index, y=y2_mean, mode='lines+markers', name=linha2,
                            line=dict(dash='solid', color='#E61C83'),  # Azul brilhante
                            error_y=dict(type='data', array=y2_std, visible=True)))

    fig.add_trace(go.Scatter(x=index, y=y3_mean, mode='lines+markers', name=linha3,
                            line=dict(dash='solid', color='#1CCEE6'),  # Roxo
                            error_y=dict(type='data', array=y3_std, visible=True)))

    # fig.add_trace(go.Scatter(x=index, y=y4_mean, mode='lines+markers', name=linha4,
    #                         line=dict(dash='solid', color='#E6CF1B'),  # Laranja
    #                         error_y=dict(type='data', array=y4_std, visible=True)))


        
    # Definição dos pontos específicos e labels
    specific_x_values = [200, 400, 600, 800]  # Supondo que estes valores estão na escala do índice
    labels = ['1x', '2x', '3x', '4x']
    
    # Adição das linhas tracejadas e labels
    # Encontrar o valor máximo entre todas as médias para definir um fim lógico para as linhas tracejadas
    # max_y_value = max(y2_mean.max(), y3_mean.max(), y4_mean.max())
    max_y_value = max(y2_mean.max(), y3_mean.max())

    # for x, label in zip(specific_x_values, labels):
    #     # Usar max_y_value * algum fator (por exemplo, 1.1) para garantir que as linhas se estendam além dos pontos mais altos
    #     fig.add_shape(type="line", x0=x, y0=0, x1=x, y1=max_y_value * 1.1, line=dict(dash="dash", color="grey"))
    #     fig.add_annotation(x=x, y=max_y_value * 1.1, text=label, showarrow=True, arrowhead=0)

    # Atualizar layout do gráfico
    fig.update_layout(
        yaxis=dict(
            showline=True, 
            showgrid=True, 
            title=yaxis_title,
            gridcolor='lightgray', 
            gridwidth=2
        ),
        xaxis=dict(
            showline=True, 
            showgrid=True, 
            title=adjusted_xaxis_title,
            gridcolor='lightgray', 
            gridwidth=2
        ),
        legend_title=None,
        margin=dict(l=120, r=10, b=100, t=25),
        autosize=True,
        width=700,
        height=600,
        template="plotly_white",
        legend=dict(x=0.1, y=1.14, traceorder='normal', orientation='h', itemwidth=30),
        font=dict(family="Arial", size=35, color="Black")
    )

    # if yaxis_title == "Taxa de Aceitação (%)":
    #     fig.update_layout(
    #         yaxis=dict(
    #             range=[75, 100],  # Set the y-axis range
    #             showgrid=True, 
    #             title=yaxis_title,
    #             gridcolor='lightgray', 
    #             gridwidth=2
    #         ),
    #     )
    
    title = yaxis_title.split(" ")[0] + ".pdf"
    fig.write_image(title)
    fig.show()  # Uncomment this line if you want to display the plot in an interactive environment


In [88]:
# ! pip install -U kaleido

In [89]:
import sys, plotly
print("Python:", sys.executable)
print("Plotly:", plotly.__version__)
try:
    import kaleido, pathlib
    print("Kaleido:", kaleido.__version__, pathlib.Path(kaleido.__file__).parent)
except Exception as e:
    print("Erro ao importar kaleido:", e)


Python: /home/davidgn/anaconda3/envs/sfc_env/bin/python
Plotly: 6.0.1
Erro ao importar kaleido: No module named 'kaleido'


In [90]:
# # Iterando sobre cada métrica para plotar
# dados_alg = []
# dados_plotagem = []
# for titulo, coluna in metricas.items():
#     # Multiplicar por 100 quando necessário para converter em porcentagem
#     multiplicador = 100 if "%" in titulo else 1

#     # Preparando dados para plotagem
#     dados_plotagem = []
#     # for alg in ['OSCIM','Resilient-OSCIM']:
#     for alg in ['GreedyB', 'GreedyB']:
#         dados_alg = dados_processados[alg][coluna] * multiplicador
#         dados_plotagem.append(dados_alg)

#     # Chamada para a função de plotagem com os dados preparados
#     plotly_three_lines_graph_with_error_bars(*dados_plotagem,
#                              yaxis_title=titulo,
#                              linha2=legendas['GreedyB'],
#                              linha3=legendas['GreedyB'],
#                             #  linha3=legendas['Resilient-OSCIM'],
#                             steps=100)

In [91]:
percentage_metrics = [
    "success",
    "bandwidth_utilization",
    "cpu_utilization",
    "cache_utilization"
]

factor = 100  # Factor by which to multiply the values

# Iterate over each algorithm (key) and its DataFrame (value) in the dictionary
for algorithm, df in dados_processados.items():
    # Check if the DataFrame contains the columns you're interested in
    cols_to_multiply = [col for col in percentage_metrics if col in df.columns]
    # Multiply the specified columns by the factor
    df[cols_to_multiply] = df[cols_to_multiply].apply(lambda x: x * factor)

In [92]:
dados_processados['Kuririn PPO']

,cpu_utilization,cache_utilization,bandwidth_utilization,success,latency,decision_time_ms,running_sfcs,cpu_saved,shared_vnfs,CPU Salva por SFC,CPU Salva por Servidor,eficiencia de cpu,eficiencia de banda,eficiencia de cache,pontuacao_latencia,pontuacao_success,QoS
tempo,,,,,,,,,,,,,,,,,
0,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
1,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
2,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
3,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
4,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,38.460606,22.791667,20.120720,91.347242,10.964023,8.565985,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,9.134724,8.134724
996,37.818106,22.854167,19.681856,91.357780,10.883568,8.331587,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,9.135778,8.135778
997,37.918333,22.979167,19.637197,91.343262,10.927716,8.237002,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,9.134326,8.134326


In [93]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import numpy as np

# def plot_metrics_in_subplots(y_data, xaxis_title='Time (s)', metrics_list=[{'Metric 1': 'm1', 'Metric 2': 'm2', 'Metric 3': 'm3'}], legends={'alg1': 'Algorithm 1', 'alg2': 'Algorithm 2', 'alg3': 'Algorithm 3'}, steps=1, font_size=14, titles=["title1", "title2"], layout_options=None):
#     if layout_options is None:
#         layout_options = {}
    
#     layout_config = {
#         'height': 900,
#         'width': 1200,
#         'legend_x': 1,
#         'legend_y': 1,
#         'legend_xanchor': 'right',
#         'legend_yanchor': 'top',
#         'legend_border': 1.1
#     }
    
#     layout_config.update(layout_options)
    
#     fig = make_subplots(rows=len(metrics_list), cols=len(metrics_list[0]), horizontal_spacing=0.11, vertical_spacing=0.13)
    
#     colors = ['#EF553B', '#00CC96', '#FFA15A']
#     dashes = ['dash', 'dot', 'solid']

#     added_legend = set()

#     for row, metrics in enumerate(metrics_list, 1):
#         for col, (metric_name, metric_key) in enumerate(metrics.items(), 1):
#             for alg, color, dash in zip(legends.keys(), colors, dashes):
#                 y = y_data[alg][metric_key]
#                 y = y.groupby(np.arange(len(y))//steps).mean()
#                 x = np.arange(len(y)) * steps
                
#                 show_legend = alg not in added_legend
#                 added_legend.add(alg)

#                 fig.add_trace(
#                     go.Scatter(x=x, y=y, mode='lines', name=legends[alg] if show_legend else None, line=dict(color=color, dash=dash), showlegend=show_legend),
#                     row=row, col=col
#                 )
    
#     fig.update_layout(
#         height=layout_config['height'],
#         width=layout_config['width'],
#         showlegend=True,
#         legend_tracegroupgap=50,
#         legend=dict(
#             x=layout_config['legend_x'],
#             y=layout_config['legend_y'],
#             xanchor=layout_config['legend_xanchor'],
#             yanchor=layout_config['legend_yanchor'],
#             borderwidth=layout_config['legend_border'],
#             orientation='h',
#             itemsizing="constant", itemwidth=70
#          ),
#         template="plotly_white",
#         margin=dict(l=40, r=40, t=40, b=40),
#         font=dict(size=font_size)
#     )

#     tick_values = [0, 200, 400, 600, 800, 1000]
#     for row, metrics in enumerate(metrics_list, 1):
#         for col, metric_name in enumerate(metrics.keys(), 1):
#             tickformat = ".1f" if metric_name == "Decision Time (s)" else ".1f"
#             tickformat = "d" if metric_name == "Bandwidth Utilization (%)" or metric_name == "CPU Utilization (%)"or metric_name == "Cache Utilization (%)" or metric_name ==  "Acceptance Ratio (%)" else ".1f"
#             fig.update_xaxes(title_text=xaxis_title, row=row, col=col, tickvals=tick_values, ticktext=[str(value) for value in tick_values], range=[0, 1000], title_font=dict(size=font_size), tickfont=dict(size=font_size))
#             fig.update_yaxes(title_text=metric_name, row=row, col=col, title_font=dict(size=font_size), tickfont=dict(size=font_size), tickformat=tickformat)

#     fig.show()

#     title = titles[0].split(" ")[0] + ".pdf"
#     fig.write_image(title) 


# legendas = {
#      'GreedyB': 'GreedyB       ',
#     #  'musfico': 'MusFiCO         ',
#     #  'gr': 'MasCo'
# }

# # Exemplo de uso da função ajustada com os dados específicos fornecidos
# metricas_1 = {
#     "Acceptance Ratio (%)": "success",
#     "Decision Time (s)": "decision_time_ms",
#     "latency (s)": "latency"
# }

# # Exemplo de uso da função ajustada com os dados específicos fornecidos
# metricas_2 = {
#     "Bandwidth Utilization (%)": "bandwidth_utilization",
#     "CPU Utilization (%)": "cpu_utilization",
#     "Cache Utilization (%)": "cache_utilization"
# }


# # Define layout options for customization
# layout_options = {
#     'height': 900,  # Example: Change plot height
#     'width': 1600,   # Example: Change plot width
#     'legend_x':0.5,  # Keep legend on the right
#     'legend_y': 1.1,  # Keep legend at the top
#     'legend_xanchor': 'center',  # Anchor legend to the right side
#     'legend_yanchor': 'top',    # Anchor legend to the top
#     'legend_border':2,
#     'legend_width':130,
# }

# # Call the function with the data and layout options
# plot_metrics_in_subplots(
#     y_data=dados_processados,
#     xaxis_title='Time (s)',
#     metrics_list=[metricas_1, metricas_2],
#     legends=legendas,
#     steps=10,
#     font_size=30,  # Adjust font size if needed
#     titles=["Network Metrics", "Resource Utilization"],
#     layout_options=layout_options  # Pass the layout options here
# )


# Gráficos BoxPlot

In [94]:
import re
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go


def plotly_boxplot_graph(
    y2: Optional[Sequence]=None,
    y3: Optional[Sequence]=None,
    y4: Optional[Sequence]=None,
    xaxis_title: str = 'Tempo (s)',
    yaxis_title: str = 'Y Axis',
    linha2: str = 'linha2',
    linha3: str = 'linha3',
    linha4: str = 'linha4',
    steps: Optional[int] = 200,
    fill_missing: bool = True,
    bfill_leading: bool = True,
    show: bool = True,
    pdf_filename: Optional[str] = None,
    width: int = 950,
    height: int = 600
):
    """
    Desenha boxplots por janelas (de tamanho `steps`) para até 3 séries.
    - Converte as séries para float, coage valores inválidos para NaN.
    - Opcionalmente faz forward-fill (ffill) e backfill apenas para NaNs iniciais.
    - Tolera séries vazias/todas NaN (são ignoradas com aviso).
    - Ordena o eixo X numericamente (categorias são labels da janela).
    - Exporta PDF (via kaleido) com fallback em HTML.

    Retorna: (fig, df_plot) — a figura e o DataFrame empilhado usado no gráfico.
    """

    def _as_series(y) -> pd.Series:
        if y is None:
            return pd.Series(dtype="float64")
        # Transformar para Series float, coagir strings etc.
        s = pd.Series(y, dtype="float64")
        s = pd.to_numeric(s, errors="coerce")
        return s

    def _window_labels(n: int, step: Optional[int]) -> np.ndarray:
        """Cria labels de janela. Ex.: steps=200 -> 200, 400, 600, ..."""
        if n == 0:
            return np.array([], dtype=int)
        if step is None or (isinstance(step, (int, np.integer)) and step <= 0):
            # Sem janelas => um label por ponto (0..n-1)
            return np.arange(n, dtype=int)
        # Agrupar em blocos de 'step' (última janela pode ser menor)
        groups = np.repeat(np.arange((n + step - 1)//step), step)[:n]
        return (groups + 1) * step

    # Preparar as três séries (opcionais)
    series_info = [
        (_as_series(y2), linha2),
        (_as_series(y3), linha3),
        (_as_series(y4), linha4),
    ]

    frames = []
    warnings = []

    for s, label in series_info:
        if s.empty:
            continue

        # Coerção/Preenchimentos
        if fill_missing:
            # Forward fill
            s = s.ffill()
            # Se ainda existir NaN (tipicamente no início), e permitido, bfill só nos líderes
            if bfill_leading and s.isna().any():
                s = s.bfill()

        # Se continuar tudo NaN, pula
        if s.dropna().empty:
            warnings.append(f"[AVISO] Série '{label}' ignorada (todos os valores são NaN).")
            continue

        # Remover NaNs remanescentes (pontos perdidos que não foram preenchidos)
        s = s.dropna()

        # Labels de tempo/janela
        time_labels = _window_labels(len(s), steps)

        frames.append(pd.DataFrame({
            xaxis_title: time_labels,
            yaxis_title: s.values,
            "alg": label
        }))

    if not frames:
        # Nada para plotar
        print("[ERRO] Nenhuma série válida para plotar (vazia ou somente NaN).")
        return None, pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)

    # Mensagens de aviso (se houver)
    for msg in warnings:
        print(msg)

    # Garantir ordenação numérica do eixo X (por padrão, categorias podem vir desordenadas)
    # Transformar a coluna do eixo X para tipo categórico com ordem crescente
    if not np.issubdtype(np.array(df[xaxis_title]).dtype, np.number):
        # Tentar coerção numérica, caso alguém tenha passado labels não numéricos
        df[xaxis_title] = pd.to_numeric(df[xaxis_title], errors="coerce")
    # Se houver NaNs após coerção (não deveria), remove linhas quebradas
    df = df.dropna(subset=[xaxis_title, yaxis_title])

    # --- Template & Plot ---
    # Evita registrar o mesmo template várias vezes
    if "draft" not in pio.templates:
        pio.templates["draft"] = go.layout.Template(
            layout_annotations=[dict(xref="paper", yref="paper", showarrow=True)]
        )

    fig = px.box(df, x=xaxis_title, y=yaxis_title, color="alg")
    fig.update_traces(quartilemethod="exclusive")

    # Ordena as categorias do X numericamente
    unique_x = sorted(df[xaxis_title].unique())
    fig.update_xaxes(categoryorder="array", categoryarray=unique_x)

    fig.update_layout(
        yaxis=dict(showline=True, showgrid=True),
        xaxis=dict(showline=True, showgrid=True),
        legend_title=None,
        margin=dict(l=100, r=10, b=80, t=25),
        autosize=True, width=width, height=height,
        template="draft",
        legend=dict(x=0.05, y=1.2, traceorder='normal', orientation='h'),
        font=dict(family="Arial", size=18, color="Black"),
    )

    if show:
        fig.show(renderer="browser")

    # --- Exportação ---
    # Nome do PDF/HTML
    safe_base = re.sub(r"[^\w\-]+", "_", yaxis_title).strip("_") or "boxplot"
    pdf_out = pdf_filename or f"{safe_base}.pdf"
    html_out = f"{safe_base}.html"

    try:
        fig.write_image(pdf_out)  # requer kaleido
        # print(f"Exportado PDF: {pdf_out}")
    except Exception as e:
        print(f"Não foi possível exportar PDF ({e}). Salvando HTML como fallback.")
        fig.write_html(html_out, include_plotlyjs="cdn")
        # print(f"Exportado HTML: {html_out}")

    return fig, df


In [95]:
dados_processados['Kuririn PPO']

,cpu_utilization,cache_utilization,bandwidth_utilization,success,latency,decision_time_ms,running_sfcs,cpu_saved,shared_vnfs,CPU Salva por SFC,CPU Salva por Servidor,eficiencia de cpu,eficiencia de banda,eficiencia de cache,pontuacao_latencia,pontuacao_success,QoS
tempo,,,,,,,,,,,,,,,,,
0,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
1,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
2,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
3,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
4,5.464122,3.065878,2.223581,100.000000,9.786588,13.921517,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,10.000000,9.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,38.460606,22.791667,20.120720,91.347242,10.964023,8.565985,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,9.134724,8.134724
996,37.818106,22.854167,19.681856,91.357780,10.883568,8.331587,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,9.135778,8.135778
997,37.918333,22.979167,19.637197,91.343262,10.927716,8.237002,0.0,0.0,0.0,NaN,0.0,inf,inf,inf,-1.0,9.134326,8.134326


In [96]:
# Dicionário com os títulos das métricas e os nomes das colunas correspondentes
metricas = {
    "CPU Utilization(%)": "cpu_utilization",
    "Cache Utilization(%)": "cache_utilization",
    "Bandwidth Utilization(%)": "bandwidth_utilization",
    "Latency (ms)": "latency",
    "Acceptance Ratio (%)": "success",
    "Decision Time (s)": "decision_time_ms"
}

# Nomes dos algoritmos para legendas
legendas = {
    # 'GreedyB': 'GreedyB',
    'MSF': 'MSF',
    'Kuririn PPO': 'Kuririn PPO',
    # 'ga': 'Genetic Algorithm',
    # 'dp': 'Dynamic Programming'
}

# Iterando sobre cada métrica para plotagem
for titulo, coluna in metricas.items():
    # Determinar se é necessário converter os valores para porcentagem
    multiplicador = 1 if "%" in titulo else 1

    # Preparando dados para a plotagem
    dados_plotagem = []
    # for alg in ['msf', 'ga', 'gr']:
    for alg in ['MSF', 'Kuririn PPO']:
        dados_alg = dados_processados[alg][coluna] * multiplicador
        dados_plotagem.append(dados_alg)

    # Chamada à função de plotagem com os dados preparados
    # plotly_boxplot_graph(*dados_plotagem,
    #                      yaxis_title=titulo,
    #                      linha2=legendas['Kuririn PPO'],
    #                      linha3=legendas['MSF'],
    #                      linha4=legendas['GreedyB'])
    
    plotly_boxplot_graph(*dados_plotagem,
                        yaxis_title=titulo,
                        linha2=legendas['Kuririn PPO'],
                        linha3=legendas['MSF'],
                        linha4=legendas['MSF'],
                        )



Não foi possível exportar PDF (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
). Salvando HTML como fallback.
Não foi possível exportar PDF (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
). Salvando HTML como fallback.
Não foi possível exportar PDF (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
). Salvando HTML como fallback.
Não foi possível exportar PDF (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
). Salvando HTML como fallback.
Não foi possível exportar PDF (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
). Salvando HTML como fallback.
Não foi possíve